# Loan Sanction Prediction â€” End-to-End Data Science Project

## Project Overview

This project focuses on analyzing historical loan application data and building a machine learning model to predict whether a loan application will be approved.

The project will cover the complete data science workflow:

1. Data understanding
2. Data cleaning
3. Exploratory data analysis (EDA)
4. Feature engineering
5. SQL analysis
6. Machine learning model development
7. Model evaluation and comparison
8. Prediction on unseen data
9. Model interpretation
10. Power BI dashboard
11. Project documentation for GitHub

### Dataset

The dataset is divided into two files:

- `loan_sanction_train.csv` â€” historical loan applications containing the target variable `Loan_Status`.
- `loan_sanction_test.csv` â€” loan applications without the target variable, which will be used for final predictions.

### Machine Learning Objective

The objective is to build a supervised machine learning classification model that predicts whether a loan application will be approved (`Y`) or rejected (`N`).

In [101]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

train = pd.read_csv("C:/Users/user/Desktop/Loan Application/Data/loan_sanction_train.csv")
test = pd.read_csv("C:/Users/user/Desktop/Loan Application/Data/loan_sanction_test.csv")

In [102]:
train.shape

(614, 13)

In [103]:
test.shape

(367, 12)

In [104]:
train.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [105]:
test.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,LP001015,Male,Yes,0,Graduate,No,5720,0,110.0,360.0,1.0,Urban
1,LP001022,Male,Yes,1,Graduate,No,3076,1500,126.0,360.0,1.0,Urban
2,LP001031,Male,Yes,2,Graduate,No,5000,1800,208.0,360.0,1.0,Urban
3,LP001035,Male,Yes,2,Graduate,No,2340,2546,100.0,360.0,NaN,Urban
4,LP001051,Male,No,0,Not Graduate,No,3276,0,78.0,360.0,1.0,Urban


## 1. Dataset Structure

The training dataset contains historical loan applications along with their loan approval status.

The test dataset contains similar applicant information but does not contain `Loan_Status`. This allows the trained machine learning model to generate predictions for unseen applications.

The training dataset contains 614 records and 13 columns, while the test dataset contains 367 records and 12 columns.

In [106]:
train.columns

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='str')

## 2. Target Variable

The target variable for this project is `Loan_Status`.

It represents the final loan decision:

- `Y` â€” Loan approved
- `N` â€” Loan rejected

The remaining applicant characteristics will be used as predictor variables to train the machine learning models.

In [107]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 62.5 KB


In [108]:
train.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [109]:
train.isna().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

## 3. Missing Value Investigation

The training dataset contains missing values in several predictor variables. Before imputing these values, the missing records will be investigated to understand their distribution and determine an appropriate treatment strategy.

The target variable, `Loan_Status`, contains no missing values, which means all training records have a known outcome.

In [110]:
missing_rows = train[train.isna().any(axis=1)]

missing_rows.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
11,LP001027,Male,Yes,2,Graduate,NaN,2500,1840.0,109.0,360.0,1.0,Urban,Y
16,LP001034,Male,No,1,Not Graduate,No,3596,0.0,100.0,240.0,NaN,Urban,Y
19,LP001041,Male,Yes,0,Graduate,NaN,2600,3500.0,115.0,NaN,1.0,Urban,Y
23,LP001050,NaN,Yes,2,Not Graduate,No,3365,1917.0,112.0,360.0,0.0,Rural,N


In [111]:
missing_rows.shape

(134, 13)

In [112]:
missing_rows.isna().sum().sort_values(ascending=False)

Credit_History       50
Self_Employed        32
LoanAmount           22
Dependents           15
Loan_Amount_Term     14
Gender               13
Married               3
Education             0
Loan_ID               0
CoapplicantIncome     0
ApplicantIncome       0
Property_Area         0
Loan_Status           0
dtype: int64

In [113]:
missing_rows.isna().sum(axis=1).value_counts().sort_index()

1    121
2     11
3      2
Name: count, dtype: int64

In [114]:
categorical_cols = [
    'Gender',
    'Married',
    'Dependents',
    'Education',
    'Self_Employed',
    'Property_Area'
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(train[col].value_counts(dropna=False))


--- Gender ---
Gender
Male      489
Female    112
NaN        13
Name: count, dtype: int64

--- Married ---
Married
Yes    398
No     213
NaN      3
Name: count, dtype: int64

--- Dependents ---
Dependents
0      345
1      102
2      101
3+      51
NaN     15
Name: count, dtype: int64

--- Education ---
Education
Graduate        480
Not Graduate    134
Name: count, dtype: int64

--- Self_Employed ---
Self_Employed
No     500
Yes     82
NaN     32
Name: count, dtype: int64

--- Property_Area ---
Property_Area
Semiurban    233
Urban        202
Rural        179
Name: count, dtype: int64


In [115]:
numerical_cols = [
    'ApplicantIncome',
    'CoapplicantIncome',
    'LoanAmount',
    'Loan_Amount_Term',
    'Credit_History'
]

train[numerical_cols].describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


## 4. Missing Values and Loan Approval

Before imputing missing values, we investigate whether the absence of information is associated with the loan approval outcome.

This helps determine whether missingness is random or whether it may contain useful information about the applicant.

The target variable `Loan_Status` is used only for investigation at this stage. No imputation or model training is performed yet.

In [116]:
missing_cols = [
    'Gender',
    'Married',
    'Dependents',
    'Self_Employed',
    'LoanAmount',
    'Loan_Amount_Term',
    'Credit_History'
]

for col in missing_cols:
    print(f"\n--- {col} ---")
    
    missing = train[train[col].isna()]
    not_missing = train[train[col].notna()]
    
    print("Missing:", len(missing))
    print("Missing approval rate:")
    print(missing['Loan_Status'].value_counts(normalize=True))
    
    print("\nNot missing approval rate:")
    print(not_missing['Loan_Status'].value_counts(normalize=True))


--- Gender ---
Missing: 13
Missing approval rate:
Loan_Status
Y    0.615385
N    0.384615
Name: proportion, dtype: float64

Not missing approval rate:
Loan_Status
Y    0.688852
N    0.311148
Name: proportion, dtype: float64

--- Married ---
Missing: 3
Missing approval rate:
Loan_Status
Y    1.0
Name: proportion, dtype: float64

Not missing approval rate:
Loan_Status
Y    0.685761
N    0.314239
Name: proportion, dtype: float64

--- Dependents ---
Missing: 15
Missing approval rate:
Loan_Status
Y    0.6
N    0.4
Name: proportion, dtype: float64

Not missing approval rate:
Loan_Status
Y    0.689482
N    0.310518
Name: proportion, dtype: float64

--- Self_Employed ---
Missing: 32
Missing approval rate:
Loan_Status
Y    0.71875
N    0.28125
Name: proportion, dtype: float64

Not missing approval rate:
Loan_Status
Y    0.685567
N    0.314433
Name: proportion, dtype: float64

--- LoanAmount ---
Missing: 22
Missing approval rate:
Loan_Status
Y    0.5
N    0.5
Name: proportion, dtype: float64

N

In [117]:
pd.crosstab(
    train['Credit_History'],
    train['Loan_Status'],
    normalize='index'
)

Loan_Status,N,Y
Credit_History,,
0.0,0.921348,0.078652
1.0,0.204211,0.795789


## 5. Numerical Variable Investigation

The numerical variables are examined before imputation to identify skewness, unusual values, and the most appropriate measure for replacing missing observations.

Median-based imputation may be preferable for highly skewed variables because extreme values can strongly influence the mean.

In [118]:
train[['ApplicantIncome',
       'CoapplicantIncome',
       'LoanAmount',
       'Loan_Amount_Term']].describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term
count,614.000000,614.000000,592.000000,600.00000
mean,5403.459283,1621.245798,146.412162,342.00000
std,6109.041673,2926.248369,85.587325,65.12041
min,150.000000,0.000000,9.000000,12.00000
25%,2877.500000,0.000000,100.000000,360.00000
50%,3812.500000,1188.500000,128.000000,360.00000
75%,5795.000000,2297.250000,168.000000,360.00000
max,81000.000000,41667.000000,700.000000,480.00000


In [119]:
train['LoanAmount'].value_counts().head(20)

LoanAmount
120.0    20
110.0    17
100.0    15
187.0    12
160.0    12
128.0    11
113.0    11
130.0    10
95.0      9
96.0      9
70.0      8
115.0     8
112.0     8
125.0     7
104.0     7
135.0     7
136.0     7
150.0     7
132.0     7
158.0     6
Name: count, dtype: int64

In [120]:
train['Loan_Amount_Term'].value_counts(dropna=False).sort_index()

Loan_Amount_Term
12.0       1
36.0       2
60.0       2
84.0       4
120.0      3
180.0     44
240.0      4
300.0     13
360.0    512
480.0     15
NaN       14
Name: count, dtype: int64

In [121]:
train[train['LoanAmount'].isna()][
    ['ApplicantIncome',
     'CoapplicantIncome',
     'LoanAmount',
     'Loan_Status']
]

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Status
0,5849,0.0,NaN,Y
35,2275,2067.0,NaN,Y
63,4945,0.0,NaN,N
81,2395,0.0,NaN,Y
95,6782,0.0,NaN,N
102,13650,0.0,NaN,Y
103,4652,3583.0,NaN,Y
113,7451,0.0,NaN,Y
127,3865,1640.0,NaN,Y
202,3992,0.0,NaN,N


In [122]:
train.groupby(train['LoanAmount'].isna())[
    ['ApplicantIncome', 'CoapplicantIncome']
].median()

,ApplicantIncome,CoapplicantIncome
LoanAmount,,
False,3806.0,1221.0
True,3928.5,0.0


In [123]:
train['Loan_Amount_Term'].value_counts(dropna=False).sort_index()

Loan_Amount_Term
12.0       1
36.0       2
60.0       2
84.0       4
120.0      3
180.0     44
240.0      4
300.0     13
360.0    512
480.0     15
NaN       14
Name: count, dtype: int64

In [124]:
train['Loan_Status'].value_counts()

Loan_Status
Y    422
N    192
Name: count, dtype: int64

In [125]:
train['Loan_Status'].value_counts(normalize=True)

Loan_Status
Y    0.687296
N    0.312704
Name: proportion, dtype: float64

In [126]:
train['Loan_Amount_Term'].value_counts(dropna=False).sort_index()

Loan_Amount_Term
12.0       1
36.0       2
60.0       2
84.0       4
120.0      3
180.0     44
240.0      4
300.0     13
360.0    512
480.0     15
NaN       14
Name: count, dtype: int64

In [127]:
train[train['LoanAmount'].isna()][
    ['ApplicantIncome',
     'CoapplicantIncome',
     'LoanAmount',
     'Loan_Status']
]

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Status
0,5849,0.0,NaN,Y
35,2275,2067.0,NaN,Y
63,4945,0.0,NaN,N
81,2395,0.0,NaN,Y
95,6782,0.0,NaN,N
102,13650,0.0,NaN,Y
103,4652,3583.0,NaN,Y
113,7451,0.0,NaN,Y
127,3865,1640.0,NaN,Y
202,3992,0.0,NaN,N


In [128]:
train.groupby(train['LoanAmount'].isna())[
    ['ApplicantIncome', 'CoapplicantIncome']
].median()

,ApplicantIncome,CoapplicantIncome
LoanAmount,,
False,3806.0,1221.0
True,3928.5,0.0


In [129]:
train['Loan_Status'].value_counts(normalize=True)

Loan_Status
Y    0.687296
N    0.312704
Name: proportion, dtype: float64

In [130]:
train['Loan_Amount_Term'].value_counts(dropna=False).sort_index()

Loan_Amount_Term
12.0       1
36.0       2
60.0       2
84.0       4
120.0      3
180.0     44
240.0      4
300.0     13
360.0    512
480.0     15
NaN       14
Name: count, dtype: int64

In [131]:
train['Credit_History'].value_counts(dropna=False)

Credit_History
1.0    475
0.0     89
NaN     50
Name: count, dtype: int64

## 6. Missing Value Treatment Strategy

After investigating the missing values and their relationship with the target variable, the following treatment strategy was selected:

### Categorical Variables

- `Gender` â†’ impute with the mode.
- `Married` â†’ impute with the mode.
- `Dependents` â†’ impute with the mode.
- `Self_Employed` â†’ impute with the mode.

These variables have relatively small proportions of missing observations, and their missing groups do not show strong evidence that missingness represents a distinct category.

### Numerical Variables

- `LoanAmount` â†’ impute with the median.

The distribution of loan amounts is affected by high values, making the median more robust than the mean.

- `Loan_Amount_Term` â†’ impute with the mode.

The value `360` is overwhelmingly the most common loan term, so it is more appropriate than using the arithmetic mean.

### Credit History

`Credit_History` will not be imputed directly with `0`.

Because missing credit history has a very different approval pattern from an observed value of `0`, missing credit history will initially be preserved as a separate category so that the machine learning model can learn its effect.

This approach avoids incorrectly assuming that missing credit history means no credit history.

In [132]:
df = train.copy()
df.shape

(614, 13)

## 7. Handling Missing Categorical Values

The categorical variables `Gender`, `Married`, `Dependents`, and `Self_Employed` contain a relatively small number of missing observations.

Since their missingness does not provide strong evidence of a separate category, the missing values will be replaced using the mode of each respective variable.

In [133]:
categorical_impute = [
    'Gender',
    'Married',
    'Dependents',
    'Self_Employed'
]

for col in categorical_impute:
    df[col] = df[col].fillna(df[col].mode()[0])

In [134]:
df[categorical_impute].isna().sum()

Gender           0
Married          0
Dependents       0
Self_Employed    0
dtype: int64

## 8. Handling Missing Loan Amounts

The `LoanAmount` variable contains 22 missing observations.

Because loan amounts can be affected by extreme values, the median is used instead of the mean. Median imputation is more robust to skewed distributions and extreme observations.

In [135]:
df['LoanAmount'] = df['LoanAmount'].fillna(
    df['LoanAmount'].median()
)

In [136]:
df['LoanAmount'].isna().sum()

np.int64(0)

## 9. Handling Missing Loan Term

The `Loan_Amount_Term` variable contains 14 missing observations.

The value `360` is by far the most common loan term in the dataset. Therefore, the missing values are replaced using the mode rather than the mean.

In [137]:
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(
    df['Loan_Amount_Term'].mode()[0]
)

In [138]:
df['Loan_Amount_Term'].isna().sum()

np.int64(0)

In [139]:
df['Credit_History'].fillna(1)

0      1.0
1      1.0
2      1.0
3      1.0
4      1.0
      ... 
609    1.0
610    1.0
611    1.0
612    1.0
613    0.0
Name: Credit_History, Length: 614, dtype: float64

## 10. Handling Missing Credit History

`Credit_History` is a binary variable containing 0 and 1.

However, 50 observations have missing credit history. Investigation showed that these missing observations have a different loan approval pattern from observations where `Credit_History = 0`.

Therefore, missing credit history is treated as a separate category rather than being incorrectly classified as either 0 or 1.

The encoding is:

- `1` â†’ Positive credit history
- `0` â†’ No/negative credit history
- `-1` â†’ Credit history not available

In [140]:
df['Credit_History'] = df['Credit_History'].fillna(-1)

In [141]:
df['Credit_History'].value_counts()

Credit_History
 1.0    475
 0.0     89
-1.0     50
Name: count, dtype: int64

In [142]:
df.isna().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [143]:
df.shape

(614, 13)

In [144]:
print("Shape:", df.shape)
print("\nTotal missing values:", df.isna().sum().sum())

Shape: (614, 13)

Total missing values: 0


## 11. Duplicate Record Investigation

Duplicate records can cause problems during machine learning because repeated observations may give the model an unrealistic representation of certain patterns.

Before removing duplicates, we will first determine whether duplicate rows exist and how many there are.

The duplicate investigation will be performed on the cleaned working dataset.

In [145]:
df.duplicated().sum()

np.int64(0)

## 12. Data Type and Category Validation

Before exploratory data analysis and machine learning, the structure of each variable is examined.

This step verifies:

- The data type of each variable
- The number of unique values
- The categories present in categorical variables
- Potential inconsistencies in categorical values

Understanding these characteristics will help determine the appropriate preprocessing and encoding techniques for the machine learning models.

In [146]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             614 non-null    str    
 2   Married            614 non-null    str    
 3   Dependents         614 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      614 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         614 non-null    float64
 9   Loan_Amount_Term   614 non-null    float64
 10  Credit_History     614 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 62.5 KB


In [147]:
df.nunique().sort_values()

Gender                 2
Married                2
Self_Employed          2
Education              2
Loan_Status            2
Property_Area          3
Credit_History         3
Dependents             4
Loan_Amount_Term      10
LoanAmount           203
CoapplicantIncome    287
ApplicantIncome      505
Loan_ID              614
dtype: int64

In [148]:
categorical_cols = [
    'Gender',
    'Married',
    'Dependents',
    'Education',
    'Self_Employed',
    'Property_Area',
    'Loan_Status'
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- Gender ---
Gender
Male      502
Female    112
Name: count, dtype: int64

--- Married ---
Married
Yes    401
No     213
Name: count, dtype: int64

--- Dependents ---
Dependents
0     360
1     102
2     101
3+     51
Name: count, dtype: int64

--- Education ---
Education
Graduate        480
Not Graduate    134
Name: count, dtype: int64

--- Self_Employed ---
Self_Employed
No     532
Yes     82
Name: count, dtype: int64

--- Property_Area ---
Property_Area
Semiurban    233
Urban        202
Rural        179
Name: count, dtype: int64

--- Loan_Status ---
Loan_Status
Y    422
N    192
Name: count, dtype: int64


In [149]:
numerical_cols = [
    'ApplicantIncome',
    'CoapplicantIncome',
    'LoanAmount',
    'Loan_Amount_Term',
    'Credit_History'
]

df[numerical_cols].describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,614.000000,614.000000,614.000000
mean,5403.459283,1621.245798,145.752443,342.410423,0.692182
std,6109.041673,2926.248369,84.107233,64.428629,0.613633
min,150.000000,0.000000,9.000000,12.000000,-1.000000
25%,2877.500000,0.000000,100.250000,360.000000,1.000000
50%,3812.500000,1188.500000,128.000000,360.000000,1.000000
75%,5795.000000,2297.250000,164.750000,360.000000,1.000000
max,81000.000000,41667.000000,700.000000,480.000000,1.000000


In [150]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             614 non-null    str    
 2   Married            614 non-null    str    
 3   Dependents         614 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      614 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         614 non-null    float64
 9   Loan_Amount_Term   614 non-null    float64
 10  Credit_History     614 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 62.5 KB


In [151]:
df.nunique().sort_values()

Gender                 2
Married                2
Self_Employed          2
Education              2
Loan_Status            2
Property_Area          3
Credit_History         3
Dependents             4
Loan_Amount_Term      10
LoanAmount           203
CoapplicantIncome    287
ApplicantIncome      505
Loan_ID              614
dtype: int64

# 13. Exploratory Data Analysis

Exploratory Data Analysis (EDA) is performed to understand the distribution of the variables and identify relationships between applicant characteristics and loan approval.

The analysis focuses on:

- Loan approval distribution
- Applicant demographics
- Income distribution
- Loan amount distribution
- Credit history
- Loan amount term
- Education
- Employment status
- Property area
- Dependents
- Relationships between predictor variables and loan approval

The findings from this stage will guide feature engineering and model development.

### 13.1 Loan Approval Distribution

The distribution of the target variable `Loan_Status` is examined to determine whether the dataset is balanced or imbalanced.

In [152]:
df['Loan_Status'].value_counts()

Loan_Status
Y    422
N    192
Name: count, dtype: int64

In [153]:
df['Loan_Status'].value_counts(normalize=True) * 100

Loan_Status
Y    68.729642
N    31.270358
Name: proportion, dtype: float64

In [154]:
df['Loan_Status'].value_counts().plot(
    kind='bar',
    figsize=(6, 4), color = ['green', 'red']
)

plt.title('Loan Approval Distribution', loc = 'left', size =20)
plt.xlabel('Loan Status')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.show()

In [155]:
plt.figure(figsize=(8, 5))

plt.hist(df['ApplicantIncome'], bins=30, color = 'green')

plt.title('Applicant Income Distribution', loc = 'left', size = 20)
plt.xlabel('Applicant Income')
plt.ylabel('Frequency')

plt.show()

In [156]:
pd.crosstab(
    df['Credit_History'],
    df['Loan_Status']
).plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Credit History vs Loan Status', size = 20, loc = 'left')
plt.xlabel('Credit History')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')
plt.show()

### 13.3 Education and Loan Approval

The relationship between an applicant's education level and loan approval is examined.

Approval rates are compared between graduate and non-graduate applicants to determine whether education level is associated with loan approval.

In [157]:
pd.crosstab(
    df['Education'],
    df['Loan_Status'],
    normalize='index'
) * 100

Loan_Status,N,Y
Education,,
Graduate,29.166667,70.833333
Not Graduate,38.805970,61.194030


In [158]:
pd.crosstab(
    df['Education'],
    df['Loan_Status']
)

Loan_Status,N,Y
Education,,
Graduate,140,340
Not Graduate,52,82


In [159]:
education_status = pd.crosstab(
    df['Education'],
    df['Loan_Status']
)

education_status.plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Education vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Education')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

### 13.4 Property Area and Loan Approval

The relationship between the applicant's property area and loan approval is examined.

Approval rates are compared across rural, semiurban, and urban property areas to determine whether property location is associated with loan approval.

In [160]:
pd.crosstab(
    df['Property_Area'],
    df['Loan_Status'],
    normalize='index'
) * 100

Loan_Status,N,Y
Property_Area,,
Rural,38.547486,61.452514
Semiurban,23.175966,76.824034
Urban,34.158416,65.841584


In [161]:
pd.crosstab(
    df['Property_Area'],
    df['Loan_Status']
)

Loan_Status,N,Y
Property_Area,,
Rural,69,110
Semiurban,54,179
Urban,69,133


In [162]:
property_status = pd.crosstab(
    df['Property_Area'],
    df['Loan_Status']
)

property_status.plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Property Area vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Property Area')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

pd.crosstab(
    df['Education'],
    df['Loan_Status'],
    normalize='index'
) * 100

### Education and Loan Approval â€” Finding

Graduate applicants have a higher loan approval rate (70.83%) compared with non-graduate applicants (61.19%).

This represents a difference of approximately 9.64 percentage points. Although education appears to be associated with loan approval in the dataset, this relationship does not imply causation because other applicant characteristics may influence the loan decision.

In [163]:
plt.figure(figsize=(8, 4))

plt.boxplot(df['ApplicantIncome'])

plt.title('Applicant Income Boxplot', loc = 'left', size = 20)
plt.ylabel('Applicant Income')

plt.show()

In [164]:
df.groupby('Loan_Status')['ApplicantIncome'].median()

Loan_Status
N    3833.5
Y    3812.5
Name: ApplicantIncome, dtype: float64

In [165]:
df.groupby('Loan_Status')['ApplicantIncome'].describe()

,count,mean,std,min,25%,50%,75%,max
Loan_Status,,,,,,,,
N,192.0,5446.078125,6819.558528,150.0,2885.0,3833.5,5861.25,81000.0
Y,422.0,5384.068720,5765.441615,210.0,2877.5,3812.5,5771.50,63337.0


In [166]:
income_approved = df[df['Loan_Status'] == 'Y']['ApplicantIncome']
income_rejected = df[df['Loan_Status'] == 'N']['ApplicantIncome']

plt.figure(figsize=(8, 5))

plt.boxplot(
    [income_approved, income_rejected],
    tick_labels=['Approved', 'Rejected']
)

plt.title('Applicant Income by Loan Status', loc = 'left', size = 20)
plt.xlabel('Loan Status')
plt.ylabel('Applicant Income')

plt.show()

In [167]:
df.groupby('Loan_Status')['ApplicantIncome'].describe()

,count,mean,std,min,25%,50%,75%,max
Loan_Status,,,,,,,,
N,192.0,5446.078125,6819.558528,150.0,2885.0,3833.5,5861.25,81000.0
Y,422.0,5384.068720,5765.441615,210.0,2877.5,3812.5,5771.50,63337.0


In [168]:
df.groupby('Loan_Status')['ApplicantIncome'].median()

Loan_Status
N    3833.5
Y    3812.5
Name: ApplicantIncome, dtype: float64

In [169]:
df.groupby('Loan_Status')['ApplicantIncome'].describe()

,count,mean,std,min,25%,50%,75%,max
Loan_Status,,,,,,,,
N,192.0,5446.078125,6819.558528,150.0,2885.0,3833.5,5861.25,81000.0
Y,422.0,5384.068720,5765.441615,210.0,2877.5,3812.5,5771.50,63337.0


### Applicant Income and Loan Approval â€” Finding

Applicant income shows very similar central tendencies between approved and rejected loan applications.

The median applicant income is 3,833.5 for rejected applications and 3,812.5 for approved applications. The mean income is also similar between the two groups.

However, the income distribution is strongly right-skewed, with a maximum income of 81,000 compared with a median of approximately 3,813.

Rejected applications also show greater income variability than approved applications, as indicated by their higher standard deviation.

Overall, applicant income alone does not appear to strongly distinguish loan approval outcomes. However, its skewed distribution may warrant transformation during feature engineering.

### 13.6 Coapplicant Income Distribution

Coapplicant income is analyzed to understand its distribution, central tendency, variability, and potential extreme observations.

The variable is also compared across loan approval outcomes to determine whether coapplicant income is associated with loan approval.

In [170]:
df['CoapplicantIncome'].describe()

count      614.000000
mean      1621.245798
std       2926.248369
min          0.000000
25%          0.000000
50%       1188.500000
75%       2297.250000
max      41667.000000
Name: CoapplicantIncome, dtype: float64

In [171]:
plt.figure(figsize=(8, 5))

plt.hist(df['CoapplicantIncome'], bins=30, color = 'green')

plt.title('Coapplicant Income Distribution', size = 20, loc = 'left')
plt.xlabel('Coapplicant Income')
plt.ylabel('Frequency')

plt.show()

In [172]:
plt.figure(figsize=(8, 4))

plt.boxplot(df['CoapplicantIncome'])

plt.title('Coapplicant Income Boxplot', size = 20, loc = 'left')
plt.ylabel('Coapplicant Income')

plt.show()

In [173]:
df.groupby('Loan_Status')['CoapplicantIncome'].describe()

,count,mean,std,min,25%,50%,75%,max
Loan_Status,,,,,,,,
N,192.0,1877.807292,4384.060103,0.0,0.0,268.0,2273.75,41667.0
Y,422.0,1504.516398,1924.754855,0.0,0.0,1239.5,2297.25,20000.0


In [174]:
df.groupby('Loan_Status')['CoapplicantIncome'].median()

Loan_Status
N     268.0
Y    1239.5
Name: CoapplicantIncome, dtype: float64

In [175]:
co_income_approved = df[df['Loan_Status'] == 'Y']['CoapplicantIncome']
co_income_rejected = df[df['Loan_Status'] == 'N']['CoapplicantIncome']

plt.figure(figsize=(8, 5))

plt.boxplot(
    [co_income_approved, co_income_rejected],
    tick_labels=['Approved', 'Rejected']
)

plt.title('Coapplicant Income by Loan Status', loc = 'left', size = 20)
plt.xlabel('Loan Status')
plt.ylabel('Coapplicant Income')

plt.show()

### Coapplicant Income and Loan Approval â€” Finding

Coapplicant income shows a substantial difference in median values between approved and rejected applications.

The median coapplicant income is 1,239.5 for approved applications compared with 268.0 for rejected applications.

The variable is strongly right-skewed, with a large number of applicants having zero coapplicant income and a small number of observations with very high incomes. This causes the mean to differ substantially from the median.

Extreme values are retained at this stage because they may represent legitimate applicants rather than data errors.

Coapplicant income appears to contain potentially useful information for predicting loan approval, although its predictive contribution will be evaluated alongside the other variables during model development.

### 13.7 Loan Amount Distribution

The distribution of loan amounts is examined to understand its central tendency, variability, skewness, and potential extreme observations.

Loan amount is also compared across loan approval outcomes to determine whether the requested loan amount is associated with loan approval.

Extreme values will be investigated rather than automatically removed, since they may represent legitimate loan applications.

In [176]:
df['LoanAmount'].describe()

count    614.000000
mean     145.752443
std       84.107233
min        9.000000
25%      100.250000
50%      128.000000
75%      164.750000
max      700.000000
Name: LoanAmount, dtype: float64

In [177]:
plt.figure(figsize=(8, 5))

plt.hist(df['LoanAmount'], bins=30, color = 'green')

plt.title('Loan Amount Distribution', loc = 'left', size = 20)
plt.xlabel('Loan Amount')
plt.ylabel('Frequency')

plt.show()

In [178]:
plt.figure(figsize=(8, 4))

plt.boxplot(df['LoanAmount'].dropna())

plt.title('Loan Amount Boxplot', loc = 'left', size = 20)
plt.ylabel('Loan Amount')

plt.show()

In [179]:
df.groupby('Loan_Status')['LoanAmount'].median()

Loan_Status
N    128.0
Y    128.0
Name: LoanAmount, dtype: float64

In [180]:
df.groupby('Loan_Status')['LoanAmount'].describe()

,count,mean,std,min,25%,50%,75%,max
Loan_Status,,,,,,,,
N,192.0,149.890625,83.529056,9.0,102.75,128.0,173.0,570.0
Y,422.0,143.869668,84.400468,17.0,100.00,128.0,160.0,700.0


In [181]:
loan_approved = df[df['Loan_Status'] == 'Y']['LoanAmount']
loan_rejected = df[df['Loan_Status'] == 'N']['LoanAmount']

plt.figure(figsize=(8, 5))

plt.boxplot(
    [loan_approved, loan_rejected],
    tick_labels=['Approved', 'Rejected']
)

plt.title('Loan Amount by Loan Status', loc = 'left', size = 20)
plt.xlabel('Loan Status')
plt.ylabel('Loan Amount')

plt.show()

### Loan Amount and Loan Approval â€” Finding

The median loan amount is identical for approved and rejected applications at 128.

The mean loan amounts are also relatively similar, with 149.89 for rejected applications and 143.87 for approved applications.

Although the loan amount distribution contains extreme observations, these values have not been removed because they may represent legitimate loan applications.

Overall, loan amount alone does not appear to strongly distinguish approved from rejected applications based on the descriptive statistics. Its predictive contribution will therefore be evaluated together with the other applicant characteristics during model development.

### 13.8 Loan Amount Term and Loan Approval

The relationship between loan repayment term and loan approval is examined.

The distribution of loan terms is analyzed and approval rates are compared across the different repayment periods to determine whether loan term is associated with loan approval.

In [182]:
df['Loan_Amount_Term'].value_counts().sort_index()

Loan_Amount_Term
12.0       1
36.0       2
60.0       2
84.0       4
120.0      3
180.0     44
240.0      4
300.0     13
360.0    526
480.0     15
Name: count, dtype: int64

In [183]:
pd.crosstab(
    df['Loan_Amount_Term'],
    df['Loan_Status']
)

Loan_Status,N,Y
Loan_Amount_Term,,
12.0,0,1
36.0,2,0
60.0,0,2
84.0,1,3
120.0,0,3
180.0,15,29
240.0,1,3
300.0,5,8
360.0,159,367


In [184]:
pd.crosstab(
    df['Loan_Amount_Term'],
    df['Loan_Status'],
    normalize='index'
) * 100

Loan_Status,N,Y
Loan_Amount_Term,,
12.0,0.000000,100.000000
36.0,100.000000,0.000000
60.0,0.000000,100.000000
84.0,25.000000,75.000000
120.0,0.000000,100.000000
180.0,34.090909,65.909091
240.0,25.000000,75.000000
300.0,38.461538,61.538462
360.0,30.228137,69.771863


### Loan Amount Term and Loan Approval â€” Finding

Loan repayment terms are highly concentrated around 360 months, with 526 of the 614 applications having this term.

Among applicants with a 360-month loan term, 69.77% were approved and 30.23% were rejected.

Some less common loan-term categories show extreme approval rates, including 100% approval for 12-, 60-, and 120-month terms. However, these categories contain only 1â€“3 observations and therefore should not be interpreted as strong evidence of an association.

The 480-month category contains 15 observations and has a lower approval rate of 40%, but the sample size is still relatively small.

Overall, `Loan_Amount_Term` will be retained as a potential predictor, while rare categories will be treated cautiously during model development.

In [185]:
term_status = pd.crosstab(
    df['Loan_Amount_Term'],
    df['Loan_Status']
)

term_status.plot(
    kind='bar',
    figsize=(10, 5), color = ['red', 'green']
)

plt.title('Loan Amount Term vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Loan Amount Term (Months)')
plt.ylabel('Number of Applications')
plt.xticks(rotation=45)
plt.legend(title='Loan Status')

plt.show()

In [186]:
pd.crosstab(
    df['Gender'],
    df['Loan_Status'],
    normalize='index'
) * 100

Loan_Status,N,Y
Gender,,
Female,33.035714,66.964286
Male,30.876494,69.123506


In [187]:
pd.crosstab(
    df['Married'],
    df['Loan_Status'],
    normalize='index'
) * 100

Loan_Status,N,Y
Married,,
No,37.089202,62.910798
Yes,28.179551,71.820449


In [188]:
pd.crosstab(
    df['Dependents'],
    df['Loan_Status'],
    normalize='index'
) * 100

Loan_Status,N,Y
Dependents,,
0,31.388889,68.611111
1,35.294118,64.705882
2,24.752475,75.247525
3+,35.294118,64.705882


In [189]:
pd.crosstab(
    df['Self_Employed'],
    df['Loan_Status'],
    normalize='index'
) * 100

Loan_Status,N,Y
Self_Employed,,
No,31.203008,68.796992
Yes,31.707317,68.292683


In [190]:
gender_status = pd.crosstab(df['Gender'], df['Loan_Status'])

gender_status.plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Gender vs Loan Status', loc = 'left', size =20)
plt.xlabel('Gender')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

In [191]:
married_status = pd.crosstab(df['Married'], df['Loan_Status'])

married_status.plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Marital Status vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Married')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

In [192]:
dependents_status = pd.crosstab(df['Dependents'], df['Loan_Status'])

dependents_status.plot(
    kind='bar',
    figsize=(8, 5),color = ['red', 'green']
)

plt.title('Dependents vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Number of Dependents')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

In [193]:
employment_status = pd.crosstab(df['Self_Employed'], df['Loan_Status'])

employment_status.plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Self-Employment Status vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Self-Employed')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

### Categorical Features and Loan Approval

The categorical variables Gender, Married, Dependents, and Self-Employed are compared with loan approval status.

Approval rates are calculated within each category to identify potential differences in loan approval outcomes. Bar charts are also used to visually compare approved and rejected applications across the categories.

The analysis helps identify categorical variables that may contain useful predictive information for the machine learning models.

### Categorical Features and Loan Approval â€” Findings

#### Gender
Loan approval rates are similar for female and male applicants, with approval rates of 66.96% and 69.12%, respectively. This suggests that Gender does not have a strong association with loan approval in the dataset.

#### Married
Married applicants have a higher approval rate (71.82%) compared with applicants who are not married (62.91%). This represents a difference of approximately 8.91 percentage points.

#### Dependents
Approval rates vary across dependent categories. Applicants with two dependents have the highest approval rate (75.25%), while applicants with one or three or more dependents have lower approval rates (64.71%). These differences should be interpreted cautiously because category sizes differ.

#### Self-Employed
Approval rates are almost identical for self-employed (68.29%) and non-self-employed applicants (68.80%). Therefore, Self_Employed does not appear to have a strong association with loan approval based on descriptive analysis alone.